# EVA — Colab Training (T4)

**Google Drive → T4 GPU (16 GB) → обучение → Drive**

### Подготовка Drive
Создайте на Google Drive папку `FCF/` и загрузите:
- `full_corpus_encoded.npy` (172 → 425 MB) → `FCF/real_data/`
- `full_latest.pt` (22 MB, опционально — resume) → `FCF/checkpoints/symbolic/`
- `trajectory_store_full.pkl` (3 MB, опционально) → `FCF/checkpoints/symbolic/`

In [ ]:
# ─── 1. Mount Drive ───
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

In [ ]:
# ─── 2. Clone repo & copy data from Drive ───
import os, shutil

REPO = '/content/FCF'
DRIVE_DIR = '/content/drive/MyDrive/FCF'

if not os.path.exists(REPO):
    !git clone https://github.com/BlackCatSpb/FCF.git {REPO}
else:
    !cd {REPO} && git pull

os.chdir(REPO)

# Copy .npy if not present
npy_dst = f'{REPO}/real_data/full_corpus_encoded.npy'
npy_src = f'{DRIVE_DIR}/real_data/full_corpus_encoded.npy'
if os.path.exists(npy_src) and not os.path.exists(npy_dst):
    shutil.copy2(npy_src, npy_dst)
    print(f'Copied .npy ({os.path.getsize(npy_dst)/1e6:.0f} MB)')

# Copy checkpoints
for fn in ['full_latest.pt', 'full_best.pt', 'trajectory_store_full.pkl']:
    src = f'{DRIVE_DIR}/checkpoints/{fn}'
    dst = f'{REPO}/checkpoints/symbolic/{fn}'
    if os.path.exists(src) and not os.path.exists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        print(f'Copied {fn}')

print(f'Data ready: {os.path.exists(npy_dst)}')

In [ ]:
# ─── 3. Install deps ───
!pip install torch numpy scikit-learn --quiet
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.mem_get_info()[1] / 1e9:.1f} GB')

In [ ]:
# ─── 4. Create Colab config (B=32, ML=256) ───
%%writefile /content/FCF/eva/symbolic/colab_config.py
DEVICE = 'cuda'
STEPS = 100000
LR = 5e-3
B = 32
ML = 256
CKPT = '/content/FCF/checkpoints/symbolic'
DRIVE_DIR = '/content/drive/MyDrive/FCF/checkpoints'

In [ ]:
# ─── 5. Add import hook to train_full_corpus.py ───
path = '/content/FCF/train_full_corpus.py'
with open(path, 'r', encoding='utf-8') as f:
    code = f.read()

marker = "sys.stderr.reconfigure(encoding='utf-8')"
hook = marker + "\ntry:\n    from eva.symbolic.colab_config import *\nexcept ImportError:\n    pass\n"
if marker in code and 'colab_config' not in code:
    code = code.replace(marker, hook)
    with open(path, 'w', encoding='utf-8') as f:
        f.write(code)
    print('Hook added')
else:
    print('Already hooked')

In [ ]:
# ─── 6. Add Drive sync to training loop ───
sync_block = '''
    # Colab: sync to Drive every 5000 steps
    if s % 5000 == 0 and os.path.exists(DRIVE_DIR):
        os.makedirs(DRIVE_DIR, exist_ok=True)
        shutil.copy2(os.path.join(CKPT, 'full_latest.pt'), os.path.join(DRIVE_DIR, 'full_latest.pt'))
        shutil.copy2(os.path.join(CKPT, 'full_best.pt'), os.path.join(DRIVE_DIR, 'full_best.pt'))'''

with open(path, 'r', encoding='utf-8') as f:
    code = f.read()

ins = "if s % 5000 == 0:"
if sync_block.strip() not in code:
    code = code.replace(ins + "\n        ut.eval()", sync_block + "\n" + ins + "\n        ut.eval()")
    with open(path, 'w', encoding='utf-8') as f:
        f.write(code)
    print('Sync callback added')
else:
    print('Already synced')

In [ ]:
# ─── 7. Run training ───
print('>>> Starting training on Colab T4')
!python train_full_corpus.py 2>&1

In [ ]:
# ─── 8. Final sync + download best checkpoint ───
import shutil
DRIVE_SYNC = '/content/drive/MyDrive/FCF/checkpoints'
os.makedirs(DRIVE_SYNC, exist_ok=True)

for fn in ['full_latest.pt', 'full_best.pt', 'trajectory_store_full.pkl']:
    src = f'/content/FCF/checkpoints/symbolic/{fn}'
    if os.path.exists(src):
        shutil.copy2(src, f'{DRIVE_SYNC}/{fn}')
        print(f'Synced {fn}')

from google.colab import files
best = '/content/drive/MyDrive/FCF/checkpoints/full_best.pt'
if os.path.exists(best):
    files.download(best)
    print('full_best.pt downloaded')
elif os.path.exists('/content/FCF/checkpoints/symbolic/full_best.pt'):
    files.download('/content/FCF/checkpoints/symbolic/full_best.pt')
    print('full_best.pt downloaded (local copy)')